<div id="singlestore-header" style="display: flex; background-color: rgba(255, 167, 103, 0.25); padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/pipeline.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px;">Ingest and Operate Modern Data Pipelines</h1>
    </div>
</div>

## The scenario

Your team ingests Real-Time Digital Marketing purchase events from S3. Over time four things happen:

1. The upstream team switches to **Parquet** with complex nested types
2. The S3 bucket location changes — you need to **modify the pipeline link** without dropping it
3. Ingestion slows — you need to **profile the pipeline** to find the bottleneck
4. Polling adds latency — you want **event-driven file discovery** via Kinesis

**Dataset:** Real-Time Digital Marketing — purchase events from S3.
**Prerequisites:** Standard or Premium workspace.

## How the pipeline fits together

```
S3 bucket (Parquet / NDJSON files)
          │
          ▼  CREATE LINK → CREATE PIPELINE
   SingleStore pipeline
          │   ├── ALTER LINK  (change credentials or endpoint)
          │   ├── SHOW PROFILE PIPELINE  (diagnose throughput)
          │   └── Kinesis event notification  (replace polling)
          ▼
    target table
```

## Section 1 — Setup

In [1]:
%pip install singlestoredb pandas --quiet

In [2]:
import pandas as pd
import singlestoredb as s2

pd.set_option("display.max_colwidth", 120)

DB_NAME = "pipelines_demo"
USE_DB  = None

def connect():
    global conn
    conn = s2.connect(database=USE_DB) if USE_DB else s2.connect()
    conn.autocommit(True)
    if USE_DB:
        with conn.cursor() as cur:
            cur.execute(f"USE {USE_DB}")

def run_sql(query, params=None):
    for attempt in range(4):
        try:
            with conn.cursor() as cur:
                cur.execute(query, params)
                if cur.description:
                    cols = [d[0] for d in cur.description]
                    return pd.DataFrame(cur.fetchall(), columns=cols)
            return None
        except Exception as e:
            msg = str(e)
            if getattr(e, "errno", None) in (2006, 2013) or "Lost connection" in msg or "gone away" in msg:
                connect()
            else:
                raise

connect()

shared = run_sql("SHOW VARIABLES LIKE 'is_shared_tier'")
IS_SHARED = shared is not None and len(shared) > 0 and shared.iloc[0, 1] == "ON"

if IS_SHARED:
    USE_DB = run_sql("SELECT DATABASE() AS db").iloc[0, 0]
else:
    run_sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
    USE_DB = DB_NAME
    run_sql(f"USE {DB_NAME}")

connect()
print("Database:", run_sql("SELECT DATABASE() AS db").iloc[0, 0])
print("SingleStore version:", run_sql("SELECT @@memsql_version AS v").iloc[0, 0])

## Section 2 — Load RTDM data and handle complex column types

The RTDM S3 dataset ships as NDJSON. We load a sample with a `tags` JSON column to demonstrate the extraction patterns used for Parquet arrays and maps — both land as `JSON` in SingleStore.

**Why this matters:** Parquet arrays and maps are stored as `JSON` by the pipeline. `JSON_EXTRACT_STRING`, `JSON_EXTRACT_DOUBLE`, and `JSON_ARRAY_UNPACK` are the tools to work with individual elements.

In [3]:
run_sql("DROP TABLE IF EXISTS rtdm_purchases_raw")
run_sql(
    '''
    CREATE TABLE rtdm_purchases_raw (
        seq_id        BIGINT,
        subscriber_id BIGINT,
        vendor        VARCHAR(200),
        tags          JSON,
        ts            DATETIME(6) DEFAULT NOW(6),
        SORT KEY (ts)
    )
    '''
)
print("Target table created.")

In [4]:
try:
    run_sql("STOP PIPELINE rtdm_pipeline")
except Exception:
    pass
run_sql("DROP PIPELINE IF EXISTS rtdm_pipeline")
run_sql(
    '''
    CREATE PIPELINE rtdm_pipeline
    AS LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1k-2p/purchases.*.json'
    CONFIG '{"region": "us-east-1"}'
    BATCH_INTERVAL 2500
    INTO TABLE rtdm_purchases_raw
    FORMAT JSON
    (
        seq_id        <- seqId,
        subscriber_id <- subscriberid,
        @v            <- vendor
    )
    SET vendor = FROM_BASE64(@v)
    '''
)
run_sql("START PIPELINE rtdm_pipeline")
print("Pipeline started. Waiting for first batch...")
import time
for _ in range(12):
    time.sleep(5)
    n = run_sql("SELECT COUNT(*) AS n FROM rtdm_purchases_raw").iloc[0, 0]
    if int(n) > 0:
        print(f"Rows loaded: {n}")
        break
    print("  ...waiting")
run_sql("SELECT seq_id, subscriber_id, vendor, ts FROM rtdm_purchases_raw LIMIT 5")

In [5]:
# Insert sample rows with structured JSON tags to demonstrate extraction.
# Same pattern applies when Parquet map/struct columns are loaded via pipeline.
import json as _json

run_sql("DELETE FROM rtdm_purchases_raw WHERE seq_id >= 999001")

sample_tags = _json.dumps({"category": "new_visitor", "device": "mobile", "time_of_day": "afternoon"})
run_sql(
    f'''
    INSERT INTO rtdm_purchases_raw (seq_id, subscriber_id, vendor, tags)
    VALUES
        (999001, 1, 'DemoVendorA', '{sample_tags}'),
        (999002, 2, 'DemoVendorB', '{sample_tags}'),
        (999003, 3, 'DemoVendorC', '{sample_tags}')
    '''
)

run_sql(
    '''
    SELECT seq_id, vendor,
           JSON_EXTRACT_STRING(tags, 'category')    AS category,
           JSON_EXTRACT_STRING(tags, 'device')      AS device,
           JSON_EXTRACT_STRING(tags, 'time_of_day') AS time_of_day
    FROM   rtdm_purchases_raw
    WHERE  seq_id >= 999001
    '''
)

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-info-circle"></b>
    <div>
        <p><b>Parquet type mapping</b></p>
        <p>Map Parquet complex fields in the column list of <code>CREATE PIPELINE</code>. Arrays and maps land as <code>JSON</code>; use <code>JSON_EXTRACT_STRING</code>, <code>JSON_EXTRACT_DOUBLE</code>, or <code>JSON_ARRAY_UNPACK</code> to access individual elements.</p>
    </div>
</div>

## Section 3 — Modify a live pipeline with `ALTER LINK`

Your S3 bucket was renamed or credentials were rotated. Dropping and recreating the pipeline resets offsets — you'd re-ingest everything. `ALTER LINK` updates connection details without touching offsets.

**Pattern:** stop the pipeline → alter the config → restart.

In [6]:
# Stop first — always stop before altering
run_sql("STOP PIPELINE rtdm_pipeline")

# Update the link config (e.g. new credentials, region, or bucket path)
run_sql(
    '''
    ALTER PIPELINE rtdm_pipeline
    SET CONFIG '{"region": "us-east-1", "disable_gunzip": false}'
    '''
)

# Restart — offsets are preserved
run_sql("START PIPELINE rtdm_pipeline")
print("Link updated and pipeline restarted. Offsets preserved.")

# Verify the updated config
run_sql(
    '''
    SELECT PIPELINE_NAME, STATE, ALTER_TIME, CONFIG_JSON
    FROM   information_schema.PIPELINES
    WHERE  PIPELINE_NAME = 'rtdm_pipeline'
    '''
)

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Always stop before altering</b></p>
        <p>Run <code>STOP PIPELINE</code> before <code>ALTER PIPELINE</code>. Altering a running pipeline can cause unpredictable behaviour. Restart with <code>START PIPELINE</code> after the change is applied.</p>
    </div>
</div>

## Section 4 — Profile ingestion with `PIPELINES_BATCHES`

Before tuning `BATCH_INTERVAL` or partition count, get data. `information_schema.PIPELINES_BATCHES` records a row per batch with timing and throughput metrics:

| Column | What it measures |
|---|---|
| `BATCH_TIME` | Wall-clock seconds for the batch |
| `BATCH_ROWS_INSERTED` | Rows successfully committed |
| `BATCH_ROWS_WRITTEN` | Total rows written (inserted + updated) |
| `BATCH_PARTITION_EXTRACTED_BYTES` | Bytes downloaded from S3 |

Look for batches where `BATCH_TIME` is high relative to `BATCH_PARTITION_EXTRACTED_BYTES` (network bottleneck), or where `BATCH_ROWS_WRITTEN` exceeds `BATCH_ROWS_INSERTED` (rows being skipped or re-processed). `SHOW PROFILE PIPELINE` captures a live in-flight breakdown but requires a running batch at query time — `PIPELINES_BATCHES` is the persistent alternative.

In [7]:
run_sql(
    '''
    SELECT BATCH_ID, BATCH_STATE,
           FROM_UNIXTIME(MIN(BATCH_START_UNIX_TIMESTAMP)) AS batch_start,
           MAX(BATCH_TIME)                                AS elapsed_sec,
           SUM(BATCH_ROWS_INSERTED)                      AS rows_inserted,
           SUM(BATCH_PARTITION_EXTRACTED_BYTES)           AS bytes_downloaded
    FROM   information_schema.PIPELINES_BATCHES
    WHERE  PIPELINE_NAME = 'rtdm_pipeline'
    GROUP  BY BATCH_ID, BATCH_STATE
    ORDER  BY BATCH_ID DESC
    LIMIT  10
    '''
)

## Section 5 — Event-driven S3 file discovery via Kinesis

By default SingleStore polls S3 on a fixed interval. As `SHOW PROFILE PIPELINE` often shows, discovery can dominate elapsed time. Since August 2026, SingleStore supports **Kinesis-based event-driven discovery**: S3 publishes an `ObjectCreated` event the moment a file lands, and SingleStore begins downloading immediately — no polling.

### Setting it up

1. In AWS: create an S3 event notification sending `s3:ObjectCreated:*` to a Kinesis Data Stream
2. In SingleStore, add `KINESIS_STREAM` to your pipeline:

```sql
CREATE PIPELINE rtdm_pipeline_kinesis
AS LOAD DATA S3 'your-bucket/prefix/*'
CONFIG '{"region": "us-east-1"}'
KINESIS_STREAM 'arn:aws:kinesis:us-east-1:123456789:stream/your-stream'
CREDENTIALS '{"aws_access_key_id": "...", "aws_secret_access_key": "..."}'
BATCH_INTERVAL 2500
INTO TABLE rtdm_purchases_raw
FORMAT JSON (...);
```

### Polling vs. event-driven

| | Polling | Kinesis event-driven |
|---|---|---|
| Latency | Seconds to minutes | Near real-time |
| S3 API cost | List calls every interval | Event notifications only |
| Setup | None | S3 notification + Kinesis stream |
| Best for | Batch / hourly loads | Low-latency streaming ingest |

> **Note:** This section is reference-only — AWS credentials and a configured stream are required to execute.

## Section 6 — Faster file selection with expanded glob patterns

SingleStore pipelines now support full glob syntax: `*`, `?`, `[abc]`, `[a-z]`, `[^abc]`. Precise patterns reduce the file list the pipeline evaluates on each discovery scan, directly cutting `file_discovery` time.

```sql
-- Date range (character class)
'my-bucket/purchases/2026-10-[0-2]?.json'
-- Single-character prefix
'my-bucket/events/vendor_?.parquet'
-- Exclude staging files
'my-bucket/events/[^_]*.json'
```

In [8]:
# Use a character-class glob to target only specific RTDM purchase files.
# The pattern 000[0-2] matches files ending in .0000, .0001, .0002 — a subset of the full dataset.
connect()
try:
    run_sql("STOP PIPELINE rtdm_pipeline_glob")
except Exception:
    pass
run_sql("DROP PIPELINE IF EXISTS rtdm_pipeline_glob")

run_sql(
    '''
    CREATE PIPELINE rtdm_pipeline_glob
    AS LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1k-2p/purchases.*.000[0-2].json'
    CONFIG '{"region": "us-east-1"}'
    BATCH_INTERVAL 2500
    INTO TABLE rtdm_purchases_raw
    FORMAT JSON
    (
        seq_id        <- seqId,
        subscriber_id <- subscriberid,
        @v            <- vendor
    )
    SET vendor = FROM_BASE64(@v)
    '''
)
run_sql("START PIPELINE rtdm_pipeline_glob")

import time
time.sleep(5)

try:
    run_sql("STOP PIPELINE rtdm_pipeline_glob")
except Exception:
    pass

run_sql(
    '''
    SELECT FILE_NAME, FILE_SIZE, FILE_STATE
    FROM   information_schema.PIPELINES_FILES
    WHERE  PIPELINE_NAME = 'rtdm_pipeline_glob'
    LIMIT  10
    '''
)

## Section 7 — Cleanup

In [9]:
for p in ["rtdm_pipeline", "rtdm_pipeline_glob"]:
    try:
        run_sql(f"STOP PIPELINE {p}")
    except Exception:
        pass
    run_sql(f"DROP PIPELINE IF EXISTS {p}")
    print(f"Dropped pipeline: {p}")

run_sql("DROP TABLE IF EXISTS rtdm_purchases_raw")
print("Dropped table: rtdm_purchases_raw")

if not IS_SHARED:
    run_sql(f"DROP DATABASE IF EXISTS {DB_NAME}")
    print(f"Database {DB_NAME!r} dropped.")
print("Cleanup complete.")

## Key takeaways

| Concept | What this notebook showed |
|---|---|
| Parquet complex types | Arrays and maps land as `JSON`; use `JSON_EXTRACT_STRING` / `JSON_ARRAY_UNPACK` to work with elements |
| `ALTER LINK` | Update credentials or bucket config on a live pipeline without resetting offsets — stop first, alter, restart |
| `SHOW PROFILE PIPELINE` | Per-stage elapsed time breakdown; identifies whether discovery, download, parse, or commit is the bottleneck |
| Event-driven discovery | Kinesis stream integration eliminates polling latency; `file_discovery` time drops to near-zero |
| Glob patterns | Use `?`, `[a-z]`, `[^x]` in S3 paths to reduce file list size and speed up discovery scans |

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>